In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Running this notebook on: ", device)

import spateo as st
print("Last run with spateo version:", st.__version__)

# Other imports
import warnings, string
import anndata as ad
import dynamo as dyn
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Uncomment the following if running on the server
import pyvista as pv
pv.start_xvfb()

sns.set_theme(context="paper", style="ticks", font_scale=1)
warnings.filterwarnings('ignore')
# %load_ext autoreload

In [2]:
adata = st.read_h5ad('/data/work/05.cluster/FuseMap/0106/mid_hind_latent_embeddings_all_single_pretrain/dmt_leiden_20250108_1.h5ad')
adata.obs_names_make_unique()
dic = {
    '0': 'mh_sc_00',
    '9': 'mh_sc_30',
    
    '1': 'mh_sc_01',
    '4': 'mh_sc_01',
    '12': 'mh_sc_01',
    
    '2': 'mh_sc_02',
    '26': 'mh_sc_02',
    
    '3': 'mh_sc_03',
    
    '5': 'mh_sc_04',
    
    '6': 'mh_sc_05',
    '19': 'mh_sc_05',
    
    '7': 'mh_sc_06',
    '8': 'mh_sc_07',
    
    '10': 'mh_sc_08',
    
    '11': 'mh_sc_09',
    '31': 'mh_sc_09',
    
    '13': 'mh_sc_10',
    
    '14': 'mh_sc_11',
    
    '15': 'mh_sc_12',
    
    '16': 'mh_sc_13',
    
    '17': 'mh_sc_14',
    
    '18': 'mh_sc_15',
    '35': 'mh_sc_15',
    
    '20': 'mh_sc_16',
    
    '21': 'mh_sc_17',
    
    '22': 'mh_sc_18',
    
    '23': 'mh_sc_19',
    
    '24': 'mh_sc_20',
    
    '25': 'mh_sc_21',
    
    '27': 'mh_sc_22',
    '32': 'mh_sc_22',
    
    '28': 'mh_sc_23',
    
    '29': 'mh_sc_24',
    
    '30': 'mh_sc_25',
    
    '33': 'mh_sc_26',
    
    '34': 'mh_sc_27',
    
    '36': 'mh_sc_28',
    '38': 'mh_sc_28',
    
    '37': 'mh_sc_29',
}
names = [
    # '20_B03606F3G5_WT202405020032.h5ad',
 '22_B03606C4E6_WT202403310050.h5ad',
 '23_B03609A4D6_WT202404150263.h5ad',
 '27_B03610C1E3_WT202403310051.h5ad',
 '31_B03619A1D3_WT202403310052.h5ad',
 '35_B03619E4G6_WT202403310053.h5ad',
 '39_A03589A1D4_WT202403310046.h5ad',
 '43_A03590E1G4_WT202403310064.h5ad',
 '47_A03593C1F3_WT202403310068.h5ad',
 '51_B03605C2E5_WT202406020126.h5ad',
 '55_B03613E3G6_WT202403310069.h5ad',
 '59_B03612E4G6_WT202403310059.h5ad',
 '63_B03606C1E3_WT202403310061.h5ad',
 '67_A03595A1D3_WT202403310062.h5ad',
 '71_A03595A4D6_WT202403310063.h5ad',
    # '75_D03468D1E3_WT202403310066.h5ad',
    # '80_D03473D4E6_WT202403310070.h5ad',
    # '84_B03423D1E3_WT202403310065.h5ad',
]
adata = adata[adata.obs['slice_code'].isin(names)]
adata.obs['dmt_leiden_anno'] = [dic[i] for i in adata.obs['dmt_leiden']]
adata = adata[adata.obs['dmt_leiden_anno'] != 'mh_sc_30'].copy()

In [3]:
adatas = []
for i in set(adata.obs['slice_code']):
    temp = adata[adata.obs['slice_code'] == i].copy()
    temp = temp[temp.obs.sample(frac=0.1).index].copy()
    if i == '47_A03593C1F3_WT202403310068.h5ad':
        temp.obsm['align_spatial_3d'] = np.array([temp.obsm['align_spatial_2d'][:, 0],
                                                  temp.obsm['align_spatial_2d'][:, 1], 
                                                  [1187.5 for i in range(len(temp))]]).T

    # sc.pp.normalize_total(temp)
    # sc.pp.log1p(temp)
    # sc.pp.scale(temp, zero_center=False, max_value=10)
    adatas.append(temp)
adata = ad.concat(adatas)

In [4]:
colormap = {'mh_sc_29': '#a3dc1f',
 'mh_sc_00': '#5ef377',
 'mh_sc_02': '#654c0d',
 'mh_sc_27': '#aea266',
 'mh_sc_03': '#e4f36b',
 'mh_sc_23': '#2a74d1',
 'mh_sc_06': '#1426db',
 'mh_sc_25': '#e93f06',
 'mh_sc_16': '#8caec2',
 'mh_sc_28': '#db3529',
 'mh_sc_18': '#215789',
 'mh_sc_22': '#490ea7',
 'mh_sc_21': '#7a7bc3',
 'mh_sc_14': '#0d204c',
 'mh_sc_19': '#4ae86f',
 'mh_sc_20': '#efb6a4',
 'mh_sc_04': '#f55fb7',
 'mh_sc_13': '#ccb347',
 'mh_sc_26': '#b36428',
 'mh_sc_09': '#efea60',
 'mh_sc_17': '#30fac0',
 'mh_sc_12': '#cee3d7',
 'mh_sc_08': '#8a1ecb',
 'mh_sc_07': '#607f79',
 'mh_sc_15': '#3a6152',
 'mh_sc_24': '#91a6de',
 'mh_sc_10': '#779d25',
 'mh_sc_11': '#524ae3',
 'mh_sc_05': '#1164b8',
 'mh_sc_01': '#1f8071'}

In [5]:
adata.obsm['3d_align_spatial'] = adata.obsm['align_spatial_3d'][:, [0, 2, 1]].copy()
adata.obsm['3d_align_spatial'][:,2] = -adata.obsm['3d_align_spatial'][:,2]
adata.obsm['3d_align_spatial'] = adata.obsm['3d_align_spatial'] - adata.obsm['3d_align_spatial'].mean(axis = 0)

In [6]:
adata.obsm['3d_align_spatial']

array([[ -47.29598621, -325.28456603,  458.41132931],
       [-297.50816496, -325.28456603,  670.86086127],
       [ 216.66304924, -325.28456603,  -79.4576354 ],
       ...,
       [-425.27695759,  224.71543397,   -5.45089157],
       [-333.11534252,  224.71543397,  140.01786803],
       [ 256.43399621,  224.71543397, -565.56929137]])

NameError: name 'np' is not defined

In [8]:
1

1

In [9]:
1

1

In [10]:
adata.write('for_plot.h5ad')

In [18]:
pc['region_anno_rgba'][:,3].min()

1.0

In [ ]:
1

In [20]:
pc['region_anno_rgba']

pyvista_ndarray([[0.9607843, 0.9607843, 0.9607843, 0.2      ],
                 [0.9607843, 0.9607843, 0.9607843, 0.2      ],
                 [0.9607843, 0.9607843, 0.9607843, 0.2      ],
                 ...,
                 [0.9607843, 0.9607843, 0.9607843, 0.2      ],
                 [0.9607843, 0.9607843, 0.9607843, 0.2      ],
                 [0.9607843, 0.9607843, 0.9607843, 0.2      ]],
                dtype=float32)

In [ ]:
celltypes = ['mh_sc_08', 'mh_sc_29', 'mh_sc_00', 'mh_sc_02', 'mh_sc_27', 'mh_sc_03', 'mh_sc_23', 'mh_sc_06', 'mh_sc_25', 'mh_sc_16', 'mh_sc_28', 'mh_sc_18', 'mh_sc_22', 'mh_sc_21', 'mh_sc_14', 'mh_sc_19', 'mh_sc_20', 'mh_sc_04', 'mh_sc_13', 'mh_sc_26', 'mh_sc_09', 'mh_sc_17', 'mh_sc_12', 'mh_sc_07', 'mh_sc_15', 'mh_sc_24', 'mh_sc_10', 'mh_sc_11', 'mh_sc_05', 'mh_sc_01']
for celltype in celltypes:
    colormap_c = colormap.copy()
    for c in colormap_c.keys():
        if c != celltype:
            colormap_c[c] = '#f5f5f520'
        else:
            colormap_c[c] = colormap_c[c]+'ff'
    pc, plot_cmap = st.tdr.construct_pc(adata=adata.copy(), spatial_key="3d_align_spatial", groupby="dmt_leiden_anno", key_added='region_anno', colormap = colormap_c)
    opacity = [0.1 if i != celltype else 0.8 for i in adata.obs['dmt_leiden_anno']]
    pc['region_anno_rgba'][:,3] = opacity
    st.pl.three_d_plot(
        model=pc,
        key='region_anno',
        # colormap=colormap_c,
        opacity=0.1,
        model_style="points",
        jupyter="html",
        background = '#EEEEEE',
        # cpo=[cpo]
        plotter_filename = f'3d_plot/celltype/{celltype}.html'
    )

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

In [60]:
gene_list = ['EN1', 'OTX2', 'GBX2', 'FGF8', 'WNT1']
for gene in gene_list:
    pc, plot_cmap = st.tdr.construct_pc(adata=adata.copy(), spatial_key="3d_align_spatial", groupby="dmt_leiden_anno", key_added='region_anno')
    pc_index=pc.point_data["obs_index"].tolist()
    exp = adata[pc_index, gene].X.toarray().flatten()
    opacity = adata[pc_index, gene].X.toarray().flatten()/10+0.05
    st.tdr.add_model_labels(model=pc, labels=exp, key_added=gene, where="point_data",inplace=True)
    st.pl.three_d_plot(
        model=pc,
        key=gene,
        colormap="twilight",
        opacity=opacity,
        model_style="points",
        jupyter="html",
        background = '#F5F5F5',
        # cpo=[cpo]
        plotter_filename = f'3d_plot/{gene}.html'
    )

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…